# 7. Regression

Predict the exact NewsGuard score 0–100 (ridge; ModernBERT with squared error on score/100). Same split, same models for both tasks. Linear models: the penalty is chosen with 3 inner folds of the
training outlets. ModernBERT: 5 epochs, lr 3e-5, effective batch 32, last epoch kept. For every test article the
outputs are saved in `predictions/regression/<model>/`, and every fold model in `Modelli/20_FromScratch/`.

In [1]:
TASK = "regression"
MODELS = "TF-IDF,Qwen3-8B,Mistral-24B"     # "ModernBERT" runs on the GPU
N_JOBS = 16

In [2]:
# Parameters
MODELS = "ModernBERT"


In [3]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import models as MO

art, outlets = MO.load()

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [4]:
for name in MODELS.split(","):
    for k in range(1, 6):
        if name == "ModernBERT":
            MO.bert_fold(TASK, art, k, "answerdotai/ModernBERT-base", name, LOG)
        else:
            MO.linear_fold(TASK, name, art, k, X=None if name == "TF-IDF" else MO.load_embeddings(name, art),
                           n_jobs=N_JOBS, logfile=LOG)

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


2026-10-31 06:59:06  regression, ModernBERT, fold 1, epoch 1/5: loss 0.0567 (856 s)


2026-10-31 07:13:17  regression, ModernBERT, fold 1, epoch 2/5: loss 0.0181 (1707 s)


2026-10-31 07:27:30  regression, ModernBERT, fold 1, epoch 3/5: loss 0.0085 (2560 s)


2026-10-31 07:41:43  regression, ModernBERT, fold 1, epoch 4/5: loss 0.0041 (3414 s)


2026-10-31 07:56:00  regression, ModernBERT, fold 1, epoch 5/5: loss 0.0020 (4270 s)


... (25 outputs left out)

2026-10-31 12:19:06  regression, ModernBERT, fold 5, epoch 3/5: loss 0.0082 (2660 s)


2026-10-31 12:33:25  regression, ModernBERT, fold 5, epoch 4/5: loss 0.0034 (3518 s)


2026-10-31 12:47:31  regression, ModernBERT, fold 5, epoch 5/5: loss 0.0016 (4365 s)


2026-10-31 12:48:40  regression, ModernBERT, fold 5: done in 4434 s, GPU peak 10.0 GB


In [5]:
for name in MODELS.split(","):
    folder = C.PREDICTIONS / TASK / name
    if name == "ModernBERT":
        print(pd.concat([pd.read_csv(f).assign(fold=f.stem[-1]) for f in sorted(folder.glob("loss_fold*.csv"))])
              .pivot(index="epoch", columns="fold", values="loss").round(4))
    else:
        p = pd.concat([pd.read_csv(f) for f in sorted(folder.glob("penalty_fold*.csv"))])
        print(name, "chosen penalty per fold:", p[p.chosen].set_index("fold").penalty.to_dict())

fold        1       2       3       4       5
epoch                                        
1      0.0567  0.0631  0.0542  0.0631  0.0542
2      0.0181  0.0178  0.0189  0.0203  0.0182
3      0.0085  0.0079  0.0084  0.0099  0.0082
4      0.0041  0.0036  0.0034  0.0043  0.0034
5      0.0020  0.0017  0.0015  0.0020  0.0016
